In [1]:
import pandas as pd
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from imblearn.combine import SMOTEENN

In [2]:
df = pd.read_csv("/content/tel_churn.csv")

In [3]:
df.head(5)

,Unnamed: 0,SeniorCitizen,MonthlyCharges,TotalCharges,Churn,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,...,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_groups_1 - 12,tenure_groups_13 - 24,tenure_groups_25 - 36,tenure_groups_37 - 48,tenure_groups_49 - 60,tenure_groups_61 - 72
0,0,0,29.85,29.85,0,1,0,0,1,1,...,0,0,1,0,1,0,0,0,0,0
1,1,0,56.95,1889.50,0,0,1,1,0,1,...,0,0,0,1,0,0,1,0,0,0
2,2,0,53.85,108.15,1,0,1,1,0,1,...,0,0,0,1,1,0,0,0,0,0
3,3,0,42.30,1840.75,0,0,1,1,0,1,...,1,0,0,0,0,0,0,1,0,0
4,4,0,70.70,151.65,1,1,0,1,0,1,...,0,0,1,0,1,0,0,0,0,0


In [4]:
df.drop('Unnamed: 0' , axis=1)

,SeniorCitizen,MonthlyCharges,TotalCharges,Churn,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,...,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_groups_1 - 12,tenure_groups_13 - 24,tenure_groups_25 - 36,tenure_groups_37 - 48,tenure_groups_49 - 60,tenure_groups_61 - 72
0,0,29.85,29.85,0,1,0,0,1,1,0,...,0,0,1,0,1,0,0,0,0,0
1,0,56.95,1889.50,0,0,1,1,0,1,0,...,0,0,0,1,0,0,1,0,0,0
2,0,53.85,108.15,1,0,1,1,0,1,0,...,0,0,0,1,1,0,0,0,0,0
3,0,42.30,1840.75,0,0,1,1,0,1,0,...,1,0,0,0,0,0,0,1,0,0
4,0,70.70,151.65,1,1,0,1,0,1,0,...,0,0,1,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,84.80,1990.50,0,0,1,0,1,0,1,...,0,0,0,1,0,1,0,0,0,0
7039,0,103.20,7362.90,0,1,0,0,1,0,1,...,0,1,0,0,0,0,0,0,0,1
7040,0,29.60,346.45,0,1,0,0,1,0,1,...,0,0,1,0,1,0,0,0,0,0
7041,1,74.40,306.60,1,0,1,0,1,1,0,...,0,0,0,1,1,0,0,0,0,0


In [5]:
df[['Churn']]

,Churn
0,0
1,0
2,1
3,0
4,1
...,...
7038,0
7039,0
7040,0
7041,1


In [6]:
#Now creating X(Independent variable) and Y(Dependent) variables
x = df.drop('Churn' , axis=1)

In [7]:
y = df['Churn']

In [8]:
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.2)

# DECISION TREE CLASSIFIER

In [9]:
model = DecisionTreeClassifier(criterion='gini', random_state = 100, max_depth = 6, min_samples_leaf=8) #USED RA

In [10]:
model.fit(x_train , y_train)

DecisionTreeClassifier(max_depth=6, min_samples_leaf=8, random_state=100)

In [11]:
y_pred = model.predict(x_test)

In [12]:
y_pred

array([0, 0, 1, ..., 1, 0, 0])

In [13]:
model.score(x_test, y_pred)

1.0

In [14]:
 print(classification_report(y_test,y_pred, labels=[0,1]))
 #ALWAYS LOOK FOR METRICS OF MINORITY CLASS(HERE IT IS CHURNERS=1)

              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1030
           1       0.67      0.48      0.56       379

    accuracy                           0.80      1409
   macro avg       0.75      0.70      0.71      1409
weighted avg       0.78      0.80      0.78      1409



# RESAMPLING

In [15]:
#WE CAN SEE ACCURACY IS VERY LOW BECAUSE DATASET IS IMBALANCED
#SO WE WILL USE SMOTEENN FOR UPSAMPLING AND DOWNSAMPLING
sm = SMOTEENN()
X_resampled , y_resampled = sm.fit_resample(x,y)

In [16]:
#NOW SPLITTING INTO TRAIN AND TEST SET AGAIN AFTER RESAMPLING
xr_train , xr_test , yr_train , yr_test = train_test_split(X_resampled , y_resampled, test_size=0.2)

In [17]:
smote_model = DecisionTreeClassifier(criterion='gini', random_state=100, max_depth= 6, min_samples_leaf = 8)

In [18]:
smote_model.fit(xr_train,yr_train)

DecisionTreeClassifier(max_depth=6, min_samples_leaf=8, random_state=100)

In [19]:
yr_pred = smote_model.predict(xr_test)

In [20]:
print(classification_report(yr_test, yr_pred , labels=[0,1]))

              precision    recall  f1-score   support

           0       0.88      0.87      0.87       395
           1       0.91      0.92      0.92       591

    accuracy                           0.90       986
   macro avg       0.90      0.89      0.89       986
weighted avg       0.90      0.90      0.90       986



We can now see that after applying SMOTEENN there is a improvement in the metrics and overall scores.

In [21]:
print(confusion_matrix(yr_test, yr_pred))

[[342  53]
 [ 47 544]]


# RANDOM FOREST CLASSIFIER

In [22]:
from sklearn.ensemble import RandomForestClassifier

In [23]:
model_rf = RandomForestClassifier(n_estimators=100, criterion='gini' , random_state=100 , max_depth = 6, min_samples_leaf = 8)
model_rf.fit(x_train,y_train)
y_pred_rf = model.predict(x_test)

In [24]:
print(classification_report(y_test, y_pred_rf, labels=[0,1]))

              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1030
           1       0.67      0.48      0.56       379

    accuracy                           0.80      1409
   macro avg       0.75      0.70      0.71      1409
weighted avg       0.78      0.80      0.78      1409



# RESAMPLED RESULTS

In [25]:
smote_model_rf = RandomForestClassifier(n_estimators=100, criterion= 'gini' , random_state=100 , max_depth=6 , min_samples_leaf = 8)
smote_model_rf.fit(xr_train, yr_train)
yr_pred_rf = smote_model_rf.predict(xr_test)

In [26]:
print(classification_report(yr_test, yr_pred_rf, labels = [0,1]))

              precision    recall  f1-score   support

           0       0.90      0.85      0.88       395
           1       0.90      0.94      0.92       591

    accuracy                           0.90       986
   macro avg       0.90      0.89      0.90       986
weighted avg       0.90      0.90      0.90       986



WE CAN SEE AN IMPROVEMENT IN THE SCORES OF MINORITY CLASS '1'

In [27]:
print(confusion_matrix(yr_test, yr_pred_rf))

[[336  59]
 [ 37 554]]


# SAVING THE MODEL

In [28]:
import pickle

In [29]:
filename = 'model.sav'

In [30]:
pickle.dump(smote_model_rf, open(filename, 'wb'))

In [31]:
load_model = pickle.load(open(filename , 'rb'))

In [32]:
load_model.score(xr_test,yr_test)

0.9026369168356998